In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path




In [3]:
try:
    eng = engs.get_engine()
    text_qry = text(engs.load_query('qry_olos.sql'))
    df = pd.read_sql(text_qry, eng)
except Exception as e:
    print (f'Erro ao executar consulta: {e}')

In [4]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [8]:
df_today = df.copy()

df_today = df_today.groupby(['area','base_type']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
df_today['tx'] = ((df_today['atendidas'] /  df_today['tentativas']) * 100 ).round(2)
df_totay = df_today[df_today['tentativas'] > 100]
df_today.sort_values(by='tx',ascending=False)


,area,base_type,tentativas,atendidas,tx
12,Nutrição,evento,1305,59,4.52
2,Enfermagem,evento,2365,86,3.64
14,Outras Áreas,Carrinho,2275,80,3.52
4,Fisioterapia,Base Leads,14242,483,3.39
6,Fisioterapia,Material Rico,983,31,3.15
16,Psicologia,Base Leads,5257,160,3.04
11,Multi,evento,830,25,3.01
10,Medicina,evento,705,20,2.84
0,Enfermagem,Material Rico,1745,49,2.81
9,Medicina,ativa,1039,28,2.69


In [7]:
df_tab = df.copy()
df_tab = df_tab[df_tab['data'] == '2026-05-13']
df_tab = df_tab.groupby(['disposition_nivel_1']).agg(
    quantidade = ('atendidas','sum')
).reset_index()
df_tab['total'] = df_tab['quantidade'].sum()
df_tab['representativada'] = ((df_tab['quantidade'] / df_tab['total']) * 100).round(2)
df_tab.sort_values(by='quantidade',ascending=False)


,disposition_nivel_1,quantidade,total,representativada
12,IMPRODUTIVO LIGACAO CAIU,25,77,32.47
21,RETORNO AGENDADO,18,77,23.38
8,IMPRODUTIVO CAIXA POSTAL NAO ATENDE,6,77,7.79
10,IMPRODUTIVO DESLIGOU NA APRESENTACAO,6,77,7.79
14,IMPRODUTIVO LIGACAO SEM AUDIO,5,77,6.49
22,RETORNO OUTRA PLATAFORMA,5,77,6.49
16,NEGOCIACAO EM ANDAMENTO,3,77,3.90
23,SEM AFINIDADE FORA DO PUBLICO,2,77,2.60
13,IMPRODUTIVO LIGACAO FALHANDO,1,77,1.30
9,IMPRODUTIVO DESCONHECE,1,77,1.30
